In [1]:
# load test dataset
# get model
# build prompt
# generate output
# evaluate
# generate report

In [2]:
import sys
sys.path.append("../")

In [3]:
# load test dataset
from pathlib import Path
from src.llm.loading import Loader
from src.llm.prompt import system_prompt


loader = Loader(path=Path("../data/data.json"), test_k=0.2, seed=42)
tests = loader.load_test_dataset()
tests

/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['input', 'target'],
    num_rows: 10
})

In [4]:
# get model
from src.llm.models import Model


model = Model(name="Qwen/Qwen2.5-1.5B-Instruct")
tokenizer = model.tokenizer
model.name

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 6143.98it/s]


'Qwen/Qwen2.5-1.5B-Instruct'

In [5]:
# build prompt
def build_prompt(input: str) -> str:
    prompt_messages = [
            {"role": "system", "content": system_prompt.render()},
            {"role": "user", "content": input},
        ]
    return tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

In [6]:
prompt = build_prompt(tests[0]["input"])
print(prompt)

<|im_start|>system
You mask personal information in text.
Keep all non-sensitive text unchanged.
Do not paraphrase.
Return only the masked text.<|im_end|>
<|im_start|>user
Hello Mrs. Walker, can you check why my recent transfer of $250 hasn't been processed? My account number is 567890 and my email is megan.walker@example.com.<|im_end|>
<|im_start|>assistant



In [7]:
# generate output

result = []
for test in tests:
    output = model.generate(build_prompt(test["input"]))
    result.append({
        "input": test["input"],
        "prediction": output,
        "target": test["target"],
    })

len(result)

10

In [8]:
# evaluate
from src.evaluating import Evaluator

evaluator = Evaluator(
    predictions=[row["prediction"] for row in result],
    references=[row["target"] for row in result],
)

metrics = evaluator.evaluate()
metrics

{'samples': 10,
 'exact_match': 0.0,
 'masking_recall': 0.0,
 'over_masking_rate': 0.0,
 'text_preservation': 0.7911512079668614}

In [9]:
# generate report
from src.reporting import write_baseline_report

report_path = write_baseline_report(
    model_name=model.name,
    metrics=metrics,
    predictions=result,
    output_path="../reports/01_llm_baseline.md",
    preview_size=len(result)
)

print(f"Baseline report saved to {report_path}")

preview = result[:3]
for index, row in enumerate(preview, start=1):
    print()
    print(f"Sample {index}")
    print("INPUT:", row["input"])
    print("PREDICTION:", row["prediction"])
    print("TARGET:", row["target"])

Baseline report saved to ../reports/01_llm_baseline.md

Sample 1
INPUT: Hello Mrs. Walker, can you check why my recent transfer of $250 hasn't been processed? My account number is 567890 and my email is megan.walker@example.com.
PREDICTION: Hello Mrs. Walker, can you check why my recent transfer of $250 hasn't been processed? Account no: 567890 & Email: megan.walker@example.com.
TARGET: Hello [PREFIX], can you check why my recent transfer of [AMOUNT] hasn't been processed? My account number is [ACCOUNT_NUMBER] and my email is [EMAIL].

Sample 2
INPUT: Dear Ms. Garcia, I need to report a lost debit card. My phone number is (555) 789-0123 and my account number is 678901.
PREDICTION: Dear Ms. Garcia, I need to report a lost debit card. My phone number is **555** 789-0123 and my account number is **678901**.
TARGET: Dear [PREFIX], I need to report a lost debit card. My phone number is [PHONE_NUMBER] and my account number is [ACCOUNT_NUMBER].

Sample 3
INPUT: I'd like to know if there are a